# 01 · Waveforms, sample rates and spectral features

*Voice & Speech Models*, notebook 1 of 4

A speech model never hears sound. It receives numbers: a waveform sampled thousands of times per second, or features computed from that waveform. In this notebook you build the path from a sine wave to log-mel features one typed function at a time, and you finish by measuring why the features matter: a classifier that is given log-mel features recognises tones that it cannot recognise from the raw samples.

## What you will build

- `make_tone`: a sampled sine wave at a chosen sample rate;
- `alias_frequency`: the frequency you actually observe after sampling;
- `dft`: the discrete Fourier transform from its definition, checked against `np.fft.rfft`;
- `frame_signal` and `stft_power`: overlapping, windowed frames and their power spectra, which together form the short-time Fourier transform;
- `mel_filterbank` and `log_mel`: triangular filters on the mel scale and log-compressed band energies.

## Learning objectives

By the end of this notebook you will be able to:

1. relate duration, sample count and sample rate, and explain why relabelling a sample rate is not resampling;
2. predict the frequency at which a tone above the Nyquist frequency appears, and see the aliasing in the samples;
3. compute a discrete Fourier transform as a matrix product and read a spectrum in hertz;
4. frame, window and transform a signal, and describe the time–frequency trade-off of the window length;
5. build a mel filterbank and log-mel features, and explain why a classifier finds them easier than raw samples.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function in the cell provided and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Cells marked **🧪 Try it** invite you to change one value and run them again.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots. The final section trains a small classifier with the tiny tensor engine from Base Model Training, which is pasted into a hidden recap cell.
- One optional cell, tagged `visualization`, plays sounds with IPython's audio player when it is available. Nothing later depends on it.
- Nothing is downloaded and no microphone is used: every sound is generated from a formula, and every random number comes from a seeded generator.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you type against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (with `@`, broadcasting arithmetic, `relu`, `tanh`, `log_softmax`, `softmax`, indexing and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been gradient-checked. Run the cell; open it only if you want to look at the code again.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 1 · Samples, sample rate and duration

A digital waveform is a list of amplitudes $x[n]$ measured every $1/f_s$ seconds, where $f_s$ is the **sample rate** in hertz (samples per second). Sample $n$ is taken at time $t_n = n / f_s$, so a signal with $N$ samples lasts $N / f_s$ seconds. A pure tone with frequency $f$ hertz, amplitude $A$ and phase $\varphi$ radians is

$$x[n] = A \sin\!\left(2\pi f \,\frac{n}{f_s} + \varphi\right), \qquad n = 0, 1, \dots, N - 1 .$$

The result is a one-dimensional array of shape `(N,)`. At 16 kHz, one second is 16,000 samples; at 8 kHz, the rate used throughout this course, it is 8,000 samples. Mono audio has one such array; stereo audio has one per channel.

In [ ]:
sample_rate = 8000                     # samples per second
n = np.arange(8)                       # the first eight sample indices
print("sample times in ms:", np.round(1000 * n / sample_rate, 3))
print("a 0.5 s clip at 8 kHz has", int(0.5 * sample_rate), "samples; at 16 kHz it has", int(0.5 * 16000))

### ✍️ Your turn: `make_tone`

Write `make_tone(frequency, duration, sample_rate, amplitude=1.0, phase=0.0)`:

- **inputs:** the frequency in hertz, the duration in seconds, the sample rate in hertz, the amplitude, and the phase in radians;
- **output:** an array of shape `(N,)` with `N = int(round(duration * sample_rate))`, following the formula above.

Hints: `t = np.arange(N) / sample_rate` gives the sample times in seconds, and `np.sin` works on a whole array at once.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def make_tone_reference(frequency, duration, sample_rate, amplitude=1.0, phase=0.0):
    n_samples = int(round(duration * sample_rate))
    t = np.arange(n_samples) / sample_rate
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)

In [ ]:
def make_tone(frequency, duration, sample_rate, amplitude=1.0, phase=0.0):
    """A sampled sine wave of shape (int(round(duration * sample_rate)),)."""
    # ✍️ Your code here (three lines: the sample count, the sample times, the sine)
    raise NotImplementedError

In [ ]:
make_tone = practice("make_tone", make_tone, make_tone_reference, cases=[
    (440.0, 0.01, 8000),
    (1000.0, 0.002, 16000, 0.5, np.pi / 2),
    (250.0, 0.0, 8000),                      # zero duration: an empty waveform
])

**Expected behaviour.** The first case returns 80 samples (10 ms at 8 kHz), the second 32 samples starting at the peak value 0.5 because a phase of $\pi/2$ turns the sine into a cosine, and the third an empty array of shape `(0,)`. The plot below draws 5 ms of a 440 Hz tone: 40 dots, one every 0.125 ms, on top of the continuous curve they were taken from.

In [ ]:
tone = make_tone(440.0, 0.005, 8000)
fine_t = np.linspace(0, 0.005, 2000)
plt.figure(figsize=(8, 2.6))
plt.plot(1000 * fine_t, np.sin(2 * np.pi * 440 * fine_t), color="0.75", label="the continuous tone")
plt.plot(1000 * np.arange(len(tone)) / 8000, tone, "o", ms=4, label="samples at 8 kHz")
plt.xlabel("time (ms)"); plt.ylabel("amplitude"); plt.title(f"A 440 Hz tone: {len(tone)} samples in 5 ms")
plt.legend(loc="upper right"); plt.tight_layout(); plt.show()

## 2 · Aliasing: what sampling cannot see

Sampling keeps only the values at $t_n = n / f_s$, and different frequencies can produce exactly the same samples. Adding a whole multiple $k f_s$ to the frequency adds $2\pi k n$ to the angle, which a sine cannot notice:

$$\sin\!\left(2\pi (f + k f_s)\frac{n}{f_s}\right) = \sin\!\left(2\pi f \frac{n}{f_s} + 2\pi k n\right) = \sin\!\left(2\pi f \frac{n}{f_s}\right).$$

A negative frequency gives the same pattern with its sign flipped. So samples taken at rate $f_s$ can only represent frequencies from $0$ up to the **Nyquist frequency** $f_s / 2$; anything higher is folded back into that range. The frequency you observe is

$$f_{\text{alias}} = \left|\, f - f_s \cdot \operatorname{round}\!\left(f / f_s\right) \right| .$$

At $f_s = 8$ kHz, a 7 kHz tone appears as a 1 kHz tone and a 5 kHz tone appears at 3 kHz. This is why audio is low-pass filtered before it is sampled, and before its rate is reduced.

In [ ]:
high = make_tone(7000.0, 0.002, 8000)
low = make_tone(1000.0, 0.002, 8000)
print("7 kHz tone, first samples:", np.round(high[:6], 3))
print("1 kHz tone, first samples:", np.round(low[:6], 3))
print("identical apart from the sign?", np.allclose(high, -low))

### ✍️ Your turn: `alias_frequency`

Write `alias_frequency(frequency, sample_rate)` that returns the observed frequency in hertz from the formula above, as a float between `0` and `sample_rate / 2`.

Hint: `np.round(frequency / sample_rate)` finds the nearest whole multiple of the sample rate.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def alias_frequency_reference(frequency, sample_rate):
    return float(abs(frequency - sample_rate * np.round(frequency / sample_rate)))

In [ ]:
def alias_frequency(frequency, sample_rate):
    """The frequency a tone appears to have after sampling, between 0 and sample_rate / 2."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
alias_frequency = practice("alias_frequency", alias_frequency, alias_frequency_reference, cases=[
    (7000.0, 8000), (3000.0, 8000), (12000.0, 8000), (440.0, 16000),
    (16000.0, 8000),                         # a whole multiple of the rate looks like 0 Hz
])

**Expected behaviour.** 7 kHz becomes 1 kHz, 3 kHz is below the Nyquist frequency and stays where it is, 12 kHz folds to 4 kHz (exactly the Nyquist frequency), and a 16 kHz tone sampled at 8 kHz gives the same value at every sample, which looks like 0 Hz. The plot shows the samples of the 7 kHz tone lying exactly on an upside-down 1 kHz curve.

In [ ]:
fine_t = np.linspace(0, 0.002, 4000)
plt.figure(figsize=(8, 2.6))
plt.plot(1000 * fine_t, np.sin(2 * np.pi * 7000 * fine_t), color="0.8", lw=0.8, label="7 kHz, continuous")
plt.plot(1000 * fine_t, -np.sin(2 * np.pi * 1000 * fine_t), color="C1", label="1 kHz, upside down")
plt.plot(1000 * np.arange(len(high)) / 8000, high, "o", color="C0", ms=5, label="samples of the 7 kHz tone")
plt.xlabel("time (ms)"); plt.title("Sampled at 8 kHz, a 7 kHz tone is indistinguishable from a 1 kHz tone")
plt.legend(loc="upper right", fontsize=8); plt.tight_layout(); plt.show()

### 🧪 Try it: fold a frequency

Change `frequency` and run the cell again. Before you run it, predict the answer with the formula.

In [ ]:
frequency = 7000.0   # try 3000, 5000, 9000 and 12000, then run this cell again
rate = 8000
print(f"a {frequency:.0f} Hz tone sampled at {rate} Hz looks like {alias_frequency(frequency, rate):.0f} Hz")

## 3 · The discrete Fourier transform

To find out which frequencies a block of $N$ samples contains, compare it with sinusoids. The **discrete Fourier transform** (DFT) is

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-2\pi i k n / N} = \sum_{n=0}^{N-1} x[n] \cos\!\left(\frac{2\pi k n}{N}\right) \;-\; i \sum_{n=0}^{N-1} x[n] \sin\!\left(\frac{2\pi k n}{N}\right).$$

Bin $k$ measures the frequency $k f_s / N$ hertz. For a real signal the bins above $N/2$ mirror the ones below, so we keep only $k = 0, 1, \dots, \lfloor N/2 \rfloor$: that is `N // 2 + 1` bins, exactly what NumPy's `np.fft.rfft` returns. The magnitude $|X[k]|$ says how strongly frequency $k f_s / N$ is present; the angle of $X[k]$ carries its phase.

In matrix form, with the angle matrix $\theta_{kn} = 2\pi k n / N$ of shape `(N // 2 + 1, N)`, the real part is $\cos(\theta)\, x$ and the imaginary part is $-\sin(\theta)\, x$. Every output is one dot product between the signal and a sinusoid.

In [ ]:
N = 8
k = np.arange(N // 2 + 1)[:, None]     # bins as a column, shape (5, 1)
n = np.arange(N)[None, :]              # samples as a row, shape (1, 8)
angles = 2 * np.pi * k * n / N         # broadcasting gives shape (5, 8)
print("angle matrix shape:", angles.shape)
print("bin frequencies at 8 kHz:", np.arange(N // 2 + 1) * 8000 / N, "Hz")

### ✍️ Your turn: `dft`

Write `dft(x)` for a real signal `x` of shape `(N,)`:

- **output:** a tuple `(real, imag)` of two arrays, each of shape `(N // 2 + 1,)`, holding the real and imaginary parts of $X[0], \dots, X[N // 2]$;
- build the angle matrix as in the demo, then `real = np.cos(angles) @ x` and `imag = -np.sin(angles) @ x`.

Returning two real arrays rather than one complex array lets the check compare both parts, so a sign mistake in the exponent cannot slip through.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def dft_reference(x):
    N = len(x)
    k = np.arange(N // 2 + 1)[:, None]
    n = np.arange(N)[None, :]
    angles = 2 * np.pi * k * n / N
    return np.cos(angles) @ x, -np.sin(angles) @ x

In [ ]:
def dft(x):
    """The DFT of a real signal x of shape (N,): returns (real, imag), each of shape (N // 2 + 1,)."""
    # ✍️ Your code here (the bins, the sample indices, the angles, then two matrix products)
    raise NotImplementedError

In [ ]:
dft = practice("dft", dft, dft_reference, cases=[
    (make_tone(1000.0, 0.004, 8000),),             # 32 samples of a 1 kHz tone
    (np.array([1.0, 0.0, 0.0, 0.0]),),             # an impulse has a flat spectrum
    (np.random.default_rng(1).normal(size=9),),    # an odd length
])

Now compare your DFT with NumPy's fast Fourier transform, which computes the same numbers in $O(N \log N)$ operations instead of $O(N^2)$. From here on the notebook uses `np.fft.rfft` for speed.

In [ ]:
signal = np.random.default_rng(2).normal(size=64)
real, imag = dft(signal)
fast = np.fft.rfft(signal)
print("largest difference from np.fft.rfft:", float(np.max(np.abs((real + 1j * imag) - fast))))
assert np.allclose(real + 1j * imag, fast), "your DFT should agree with np.fft.rfft"

for tone_frequency in (1000.0, 7000.0):
    real, imag = dft(make_tone(tone_frequency, 256 / 8000, 8000))
    peak = int(np.argmax(np.hypot(real, imag)))
    print(f"{tone_frequency:.0f} Hz tone at 8 kHz: the spectrum peaks at bin {peak}, which is {peak * 8000 / 256:.0f} Hz")

**Expected behaviour.** The difference from `np.fft.rfft` is around $10^{-14}$, which is rounding error. Both tones peak at bin 32, which is 1000 Hz: the spectrum confirms the aliasing you predicted. The plot shows the magnitude spectrum of two tones, 1 kHz and a quieter 2.5 kHz.

In [ ]:
pair = make_tone(1000.0, 256 / 8000, 8000) + 0.5 * make_tone(2500.0, 256 / 8000, 8000)
real, imag = dft(pair)
plt.figure(figsize=(8, 2.6))
plt.plot(np.arange(len(real)) * 8000 / 256, np.hypot(real, imag))
plt.xlabel("frequency (Hz)"); plt.ylabel("|X[k]|"); plt.title("Two tones, two peaks: 1000 Hz and 2500 Hz")
plt.tight_layout(); plt.show()

## 4 · Frames, windows and the short-time Fourier transform

Speech changes every few tens of milliseconds, so one spectrum of a whole utterance would blur everything together. The **short-time Fourier transform** (STFT) cuts the signal into short, overlapping **frames**, multiplies each frame by a smooth **window**, and takes the spectrum of every frame. With frame length $L$ and hop $H$, both in samples, frame $m$ holds samples $mH, \dots, mH + L - 1$, and a signal of $N \ge L$ samples gives

$$M = 1 + \left\lfloor \frac{N - L}{H} \right\rfloor$$

complete frames, stacked into an array of shape `(M, L)`. We drop a final partial frame. Libraries often pad the signal instead so that frames are centred on their time stamps, which changes the count: always check which convention a model was trained with.

Cutting a frame out abruptly creates false high frequencies at its edges. The periodic **Hann window** $w[n] = \tfrac12 - \tfrac12 \cos(2\pi n / L)$ tapers each frame smoothly to zero. (NumPy's `np.hanning` is the symmetric variant, which divides by $L - 1$; spectral analysis uses the periodic one.) The **power spectrogram** is $|\operatorname{rfft}(w \cdot \text{frame})|^2$ for every frame, an array of shape `(M, L // 2 + 1)`.

At 8 kHz, $L = 256$ samples is 32 ms and a hop of $H = 128$ samples is 16 ms, which gives 62.5 frames per second.

In [ ]:
def hann_window(length):
    """The periodic Hann window: 0.5 - 0.5 cos(2 pi n / length) for n = 0, ..., length - 1."""
    return 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(length) / length)


print("Hann window of length 8:", np.round(hann_window(8), 3))
print("frames in 1 s at 8 kHz with L = 256, H = 128:", 1 + (8000 - 256) // 128)

### ✍️ Your turn: `frame_signal`

Write `frame_signal(x, frame_length, hop)` that returns an array of shape `(M, frame_length)` whose row `m` is `x[m * hop : m * hop + frame_length]`, with $M = 1 + \lfloor (N - L) / H \rfloor$. You may assume `len(x) >= frame_length`.

Hints: a list comprehension inside `np.array(...)` works; so does an index matrix, `np.arange(frame_length)[None, :] + hop * np.arange(M)[:, None]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def frame_signal_reference(x, frame_length, hop):
    n_frames = 1 + (len(x) - frame_length) // hop
    return np.array([x[m * hop: m * hop + frame_length] for m in range(n_frames)])

In [ ]:
def frame_signal(x, frame_length, hop):
    """Overlapping frames of x: shape (1 + (len(x) - frame_length) // hop, frame_length)."""
    # ✍️ Your code here (count the frames, then slice them out)
    raise NotImplementedError

In [ ]:
ramp = np.arange(10.0)
frame_signal = practice("frame_signal", frame_signal, frame_signal_reference, cases=[
    (ramp, 4, 2),     # frames start at samples 0, 2, 4 and 6
    (ramp, 4, 3),     # the leftover samples 9 and beyond do not fill a frame and are dropped
    (ramp, 10, 5),    # exactly one frame
    (ramp, 2, 4),     # a hop longer than the frame skips samples
])
print(frame_signal(ramp, 4, 2))

### ✍️ Your turn: `stft_power`

Write `stft_power(x, frame_length, hop)`: frame the signal with your `frame_signal`, multiply every frame by `hann_window(frame_length)`, take `np.fft.rfft(..., axis=1)` and return the squared magnitude, of shape `(M, frame_length // 2 + 1)`.

Hint: broadcasting multiplies every row by the window: `frames * hann_window(frame_length)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def stft_power_reference(x, frame_length, hop):
    frames = frame_signal(x, frame_length, hop) * hann_window(frame_length)
    return np.abs(np.fft.rfft(frames, axis=1)) ** 2

In [ ]:
def stft_power(x, frame_length, hop):
    """Power spectrogram |rfft(window * frame)|^2: shape (M, frame_length // 2 + 1)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
stft_power = practice("stft_power", stft_power, stft_power_reference, cases=[
    (make_tone(1000.0, 0.1, 8000), 256, 128),
    (np.random.default_rng(3).normal(size=300), 64, 32),
    (np.ones(16), 16, 8),            # one frame of a constant: all the power is in bin 0
])

To look at a spectrogram we need a signal whose frequency changes. A **chirp** sweeps its frequency linearly from $f_0$ to $f_1$ over $T$ seconds; its phase is the running integral of the frequency, $2\pi\left(f_0 t + (f_1 - f_0)\, t^2 / (2T)\right)$. The plot shows a chirp from 200 Hz to 3500 Hz followed by a short click, in decibels, $10 \log_{10}$ of the power.

In [ ]:
def make_chirp(f_start, f_end, duration, sample_rate):
    """A sine whose frequency rises linearly from f_start to f_end hertz."""
    t = np.arange(int(round(duration * sample_rate))) / sample_rate
    return np.sin(2 * np.pi * (f_start * t + (f_end - f_start) * t ** 2 / (2 * duration)))


def show_spectrogram(x, sample_rate, frame_length, hop, title):
    power = stft_power(x, frame_length, hop)
    duration = len(x) / sample_rate
    plt.figure(figsize=(8, 2.8))
    plt.imshow(10 * np.log10(power.T + 1e-10), origin="lower", aspect="auto", cmap="magma",
               extent=[0, duration, 0, sample_rate / 2], vmin=-60)
    plt.colorbar(label="dB"); plt.xlabel("time (s)"); plt.ylabel("frequency (Hz)"); plt.title(title)
    plt.tight_layout(); plt.show()
    return power


click = np.zeros(2400); click[1200] = 5.0
sweep = np.concatenate([make_chirp(200.0, 3500.0, 1.0, 8000), click])
power = show_spectrogram(sweep, 8000, 256, 128, "A chirp from 200 Hz to 3500 Hz, then a click")
print("spectrogram shape (frames, bins):", power.shape)

### 🧪 Try it: the window length trade-off

A frame of $L$ samples can only separate frequencies that are a few multiples of $f_s / L$ hertz apart (with a Hann window, about two to four bins), and it smears anything shorter than $L / f_s$ seconds across the whole frame. The signal below holds two close tones, 1000 Hz and 1100 Hz, and then a click. With $L = 256$ the tones are about three bins apart and only just separate, and the click is a thin vertical line. Try $L = 64$: the click becomes sharper still, but the two tones merge into one band. Try $L = 1024$: the tones separate cleanly, but the click spreads over 128 ms.

In [ ]:
frame_length = 256   # try 64 and 1024, then run this cell again (the hop stays at half a frame)
two_tones = make_tone(1000.0, 0.6, 8000) + make_tone(1100.0, 0.6, 8000)
test_signal = np.concatenate([two_tones, click])
print(f"frame = {1000 * frame_length / 8000:.0f} ms, bin spacing = {8000 / frame_length:.1f} Hz")
_ = show_spectrogram(test_signal, 8000, frame_length, frame_length // 2, f"Two tones 100 Hz apart, then a click (L = {frame_length})")

## 5 · The mel scale and log-mel features

People hear pitch roughly linearly below about 1 kHz and roughly logarithmically above it. The **mel scale** models this; we use the common HTK formula

$$m = 2595 \log_{10}\!\left(1 + \frac{f}{700}\right), \qquad f = 700\left(10^{m / 2595} - 1\right).$$

A **mel filterbank** has `n_mels` triangular filters whose centres are equally spaced in mels between `f_min` and `f_max`. Take `n_mels + 2` equally spaced mel values and convert them to hertz, giving edges $e_0 < e_1 < \dots < e_{n_\text{mels} + 1}$. Filter $j$ rises linearly from 0 at $e_j$ to 1 at $e_{j+1}$ and falls back to 0 at $e_{j+2}$:

$$F_j(f) = \max\!\left(0,\; \min\!\left(\frac{f - e_j}{e_{j+1} - e_j},\; \frac{e_{j+2} - f}{e_{j+2} - e_{j+1}}\right)\right),$$

evaluated at every FFT bin frequency $f_k = k f_s / L$ for $k = 0, \dots, L/2$. The filterbank is a matrix of shape `(n_mels, L // 2 + 1)`. Multiplying the power spectrogram by its transpose adds up the power inside each band, giving shape `(M, n_mels)`: 129 numbers per frame become 20. Finally a logarithm compresses the enormous range of power values: a sound 1000 times more powerful is only $\ln 1000 \approx 6.9$ larger, which suits a model that adds weighted inputs. A small $\varepsilon$ keeps $\log 0$ out of silent frames. (Some libraries also scale each triangle to equal area, called Slaney normalisation; here every peak has height 1.)

In [ ]:
def hz_to_mel(frequency):
    """HTK mel scale."""
    return 2595.0 * np.log10(1.0 + np.asarray(frequency, dtype=float) / 700.0)


def mel_to_hz(mel):
    """Inverse of hz_to_mel."""
    return 700.0 * (10.0 ** (np.asarray(mel, dtype=float) / 2595.0) - 1.0)


print("1000 Hz =", round(float(hz_to_mel(1000.0)), 1), "mel and 4000 Hz =", round(float(hz_to_mel(4000.0)), 1), "mel")
edges = mel_to_hz(np.linspace(hz_to_mel(0.0), hz_to_mel(4000.0), 6))
print("six equally spaced mel values, in Hz:", np.round(edges))

### ✍️ Your turn: `mel_filterbank`

Write `mel_filterbank(n_mels, n_fft, sample_rate, f_min=0.0, f_max=None)`, where `f_max=None` means `sample_rate / 2`:

- **edges:** `mel_to_hz(np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2))`;
- **bin frequencies:** `np.arange(n_fft // 2 + 1) * sample_rate / n_fft`;
- **output:** shape `(n_mels, n_fft // 2 + 1)`, whose row `j` is the triangle $F_j$ above, with continuous weights (do not round the edges to whole bins).

Hint: slice the edges into `left = edges[:-2]`, `centre = edges[1:-1]` and `right = edges[2:]`, and add `[:, None]` to each so that they broadcast against the row of bin frequencies. Then compute `rising` and `falling` and combine them with `np.maximum(0.0, np.minimum(rising, falling))`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mel_filterbank_reference(n_mels, n_fft, sample_rate, f_min=0.0, f_max=None):
    f_max = sample_rate / 2 if f_max is None else f_max
    edges = mel_to_hz(np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2))
    freqs = np.arange(n_fft // 2 + 1) * sample_rate / n_fft
    left, centre, right = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    rising = (freqs - left) / (centre - left)
    falling = (right - freqs) / (right - centre)
    return np.maximum(0.0, np.minimum(rising, falling))

In [ ]:
def mel_filterbank(n_mels, n_fft, sample_rate, f_min=0.0, f_max=None):
    """Triangular mel filters: shape (n_mels, n_fft // 2 + 1)."""
    # ✍️ Your code here (edges, bin frequencies, rising and falling slopes, then combine)
    raise NotImplementedError

In [ ]:
mel_filterbank = practice("mel_filterbank", mel_filterbank, mel_filterbank_reference, cases=[
    (4, 16, 8000),
    (20, 256, 8000),
    (6, 32, 16000, 300.0, 3000.0),   # a band-limited filterbank
])
bank = mel_filterbank(20, 256, 8000)
plt.figure(figsize=(8, 2.6))
plt.plot(np.arange(129) * 8000 / 256, bank.T)
plt.xlabel("frequency (Hz)"); plt.ylabel("weight"); plt.title("20 mel filters for 256-sample frames at 8 kHz")
plt.tight_layout(); plt.show()

**Expected behaviour.** The triangles are narrow at low frequencies and wide at high frequencies, and neighbouring triangles cross at height one half, so together they cover the whole band from 0 to 4 kHz.

### ✍️ Your turn: `log_mel`

Write `log_mel(power, filterbank, eps=1e-10)` that returns `np.log(power @ filterbank.T + eps)`, of shape `(M, n_mels)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def log_mel_reference(power, filterbank, eps=1e-10):
    return np.log(power @ filterbank.T + eps)

In [ ]:
def log_mel(power, filterbank, eps=1e-10):
    """Log band energies: (M, n_bins) power and (n_mels, n_bins) filters give (M, n_mels)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
log_mel = practice("log_mel", log_mel, log_mel_reference, cases=[
    (np.random.default_rng(4).uniform(size=(5, 9)), mel_filterbank(4, 16, 8000)),
    (np.zeros((2, 9)), mel_filterbank(4, 16, 8000)),     # silence: every value is log(eps), about -23
])

The next cell joins your functions into one feature pipeline and shows a short "melody" of three tones, 500, 850 and 1400 Hz, separated by silence. These are the three symbols that the recognisers in notebooks 2 and 3 learn to transcribe.

In [ ]:
def log_mel_spectrogram(x, sample_rate, frame_length=256, hop=128, n_mels=20):
    """Waveform (N,) -> log-mel features (M, n_mels), built from your functions."""
    return log_mel(stft_power(x, frame_length, hop), mel_filterbank(n_mels, frame_length, sample_rate))


gap = np.zeros(800)
melody = np.concatenate([gap, make_tone(500.0, 0.15, 8000, 0.6), gap, make_tone(850.0, 0.15, 8000, 0.6), gap,
                         make_tone(1400.0, 0.15, 8000, 0.6), gap])
melody = melody + 0.01 * np.random.default_rng(5).normal(size=len(melody))
features = log_mel_spectrogram(melody, 8000)
print("waveform:", melody.shape, "-> log-mel features:", features.shape)

fig, axes = plt.subplots(2, 1, figsize=(8, 4.4))
axes[0].plot(np.arange(len(melody)) / 8000, melody, lw=0.5)
axes[0].set_ylabel("amplitude"); axes[0].set_title("Waveform: three tones separated by silence")
axes[1].imshow(features.T, origin="lower", aspect="auto", cmap="magma", extent=[0, len(melody) / 8000, 0, 20])
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("mel band"); axes[1].set_title("Log-mel spectrogram")
plt.tight_layout(); plt.show()

*Optional: listen.* The cell below plays the melody and the chirp if IPython's audio player is available (it is in Jupyter, JupyterLite and Colab). Nothing later depends on it.

In [ ]:
try:
    from IPython.display import Audio, display
except ImportError:
    print("IPython's audio player is not available here; the plots above show the same sounds.")
else:
    display(Audio(melody, rate=8000))
    display(Audio(sweep, rate=8000))

## 6 · 🧪 Relabelling versus resampling

A model trained on 16 kHz audio expects 16,000 samples per second. Suppose a clip was recorded at 8 kHz. There are three things you might do:

1. **Relabel** it: keep the samples and declare 16 kHz. Each sample now stands for half as much time, so the clip lasts half as long and every frequency doubles.
2. **Resample** it: compute new samples at the new rate that describe the same sound. Going up needs interpolation; going down needs a low-pass filter first.
3. **Decimate naively**: to go from 16 kHz to 8 kHz, keep every second sample. That halves the rate without the filter, so anything between 4 and 8 kHz aliases into the new band.

The resampler below works in the frequency domain: it keeps only the frequencies that both rates can represent and transforms back. It treats the clip as one period of a repeating signal, which is fine for these steady tones; production resamplers use windowed-sinc or polyphase filters. `peak_frequency` reads the strongest frequency off the spectrum.

In [ ]:
def resample(x, rate_in, rate_out):
    """Band-limited resampling with the FFT: keep the frequencies both rates can represent."""
    n_out = int(round(len(x) * rate_out / rate_in))
    spectrum = np.fft.rfft(x)
    kept = np.zeros(n_out // 2 + 1, dtype=complex)
    count = min(len(kept), len(spectrum))
    kept[:count] = spectrum[:count]
    return np.fft.irfft(kept, n=n_out) * (n_out / len(x))


def peak_frequency(x, sample_rate):
    """The frequency of the largest peak in the spectrum of x, in hertz."""
    return float(np.argmax(np.abs(np.fft.rfft(x))) * sample_rate / len(x))


clip_8k = make_tone(440.0, 0.5, 8000)
for name, (signal, rate) in {
    "original at 8 kHz": (clip_8k, 8000),
    "relabelled as 16 kHz": (clip_8k, 16000),
    "resampled to 16 kHz": (resample(clip_8k, 8000, 16000), 16000),
}.items():
    print(f"{name:21s}: {len(signal):5d} samples, {len(signal) / rate:.3f} s, peak at {peak_frequency(signal, rate):.0f} Hz")

**Expected behaviour.** Relabelling gives 4,000 samples that last 0.25 s and peak at 880 Hz: the same numbers now describe a different sound. Resampling gives 8,000 samples that last 0.5 s and still peak at 440 Hz.

Now go down in rate. The 16 kHz clip below contains 440 Hz and a quieter component above the new Nyquist frequency of 4 kHz. Keeping every second sample folds that component into the 8 kHz band; the band-limited resampler removes it first.

In [ ]:
component = 5000.0   # try 3000.0 (below the new Nyquist frequency) and 7000.0, then run this cell again
clip_16k = make_tone(440.0, 0.5, 16000) + 0.5 * make_tone(component, 0.5, 16000)
naive = clip_16k[::2]                      # 8 kHz without a low-pass filter
proper = resample(clip_16k, 16000, 8000)   # band-limited
freqs = np.arange(len(naive) // 2 + 1) * 8000 / len(naive)
fig, axes = plt.subplots(1, 2, figsize=(9, 2.6), sharey=True)
for axis, signal, title in [(axes[0], naive, "every second sample"), (axes[1], proper, "band-limited resampling")]:
    axis.plot(freqs, np.abs(np.fft.rfft(signal)) / len(signal))
    axis.set_title(title); axis.set_xlabel("frequency (Hz)")
plt.tight_layout(); plt.show()
alias = alias_frequency(component, 8000)
near = np.abs(freqs - alias) < 20
print(f"a {component:.0f} Hz component at 8 kHz would alias to {alias:.0f} Hz")
print(f"magnitude near {alias:.0f} Hz: naive {np.abs(np.fft.rfft(naive))[near].max() / len(naive):.3f}, "
      f"resampled {np.abs(np.fft.rfft(proper))[near].max() / len(proper):.3f}")

**Expected behaviour.** With the 5 kHz component, the naive version shows a false peak at 3 kHz with height about 0.25 (half of the component's amplitude 0.5), and the resampled version shows almost nothing there. With 3 kHz the component is genuine, so both versions keep it. A model fed the naive version would hear a tone that was never in the recording.

## 7 · Put it together: why features matter

Here is the payoff. You will train the same classifier twice on the same clips: once on the raw samples and once on log-mel features. Each clip is 128 ms (1,024 samples at 8 kHz) of one of three tones, 440, 660 or 990 Hz, slightly detuned, at a random loudness and a **random phase**, with background noise. The model is softmax regression, a single dense layer trained with your tiny tensor engine; only its input changes.

A linear model on raw samples computes a weighted sum of the samples. Shift a tone's phase by half a cycle and every sample changes sign, so that sum changes sign too: no single weight vector can respond to "440 Hz" whatever the phase. The power spectrum throws the phase away and keeps the frequency, so on log-mel features the three classes become easy to separate.

In [ ]:
def tone_clips(n, rng, frequencies=(440.0, 660.0, 990.0), n_samples=1024, sample_rate=8000, noise=0.1):
    """n noisy clips of one of three tones, each with a random detuning, loudness and phase."""
    labels = rng.integers(0, len(frequencies), size=n)
    clips = np.zeros((n, n_samples))
    for i, label in enumerate(labels):
        clips[i] = make_tone(frequencies[label] * rng.uniform(0.98, 1.02), n_samples / sample_rate, sample_rate,
                             rng.uniform(0.3, 1.0), rng.uniform(0.0, 2 * np.pi))
    return clips + noise * rng.normal(size=clips.shape), labels


def train_softmax_regression(train_x, train_y, steps=300, learning_rate=0.05, seed=0):
    """One dense layer and cross-entropy, trained with full-batch gradient descent on the tiny tensor engine."""
    mean, std = train_x.mean(axis=0), train_x.std(axis=0) + 1e-8      # training statistics only
    inputs = Tensor((train_x - mean) / std)
    W = parameter(train_x.shape[1], 3, rng=np.random.default_rng(seed), scale=0.01)
    b = zeros_parameter(3)
    losses = []
    for step in range(steps):
        loss = cross_entropy(inputs @ W + b, train_y)
        zero_grad([W, b])
        loss.backward()
        for p in (W, b):
            p.data -= learning_rate * p.grad
        losses.append(float(loss.data))

    def predict(x):
        return np.argmax(((x - mean) / std) @ W.data + b.data, axis=1)

    return predict, losses


clip_rng = np.random.default_rng(42)
train_clips, train_labels = tone_clips(600, clip_rng)
test_clips, test_labels = tone_clips(300, clip_rng)
inputs = {
    "raw samples": (train_clips, test_clips),
    "log-mel": (np.array([log_mel_spectrogram(c, 8000).mean(axis=0) for c in train_clips]),
                np.array([log_mel_spectrogram(c, 8000).mean(axis=0) for c in test_clips])),
}
results = {}
for name, (train_x, test_x) in inputs.items():
    predict, losses = train_softmax_regression(train_x, train_labels)
    results[name] = {"inputs per clip": train_x.shape[1], "first loss": losses[0], "last loss": losses[-1],
                     "train accuracy": float(np.mean(predict(train_x) == train_labels)),
                     "test accuracy": float(np.mean(predict(test_x) == test_labels))}
    print(f"{name:11s}: {train_x.shape[1]:4d} inputs, loss {losses[0]:.3f} -> {losses[-1]:.3f}, "
          f"train accuracy {results[name]['train accuracy']:.3f}, test accuracy {results[name]['test accuracy']:.3f}")

In [ ]:
assert log_mel_spectrogram(np.zeros(8000), 8000).shape == (1 + (8000 - 256) // 128, 20), "1 s at 8 kHz gives 61 frames of 20 bands"
for name in results:
    assert results[name]["last loss"] < results[name]["first loss"], f"training on {name} should reduce the loss"
assert results["log-mel"]["test accuracy"] >= 0.95, "log-mel features should make the three tones easy to tell apart"
assert results["raw samples"]["test accuracy"] <= 0.60, "raw samples with random phase should defeat a linear model"
print(f"✅ Same model, same clips: {results['raw samples']['test accuracy']:.0%} from raw samples, "
      f"{results['log-mel']['test accuracy']:.0%} from your log-mel features (chance is 33%).")

**What to notice.** On raw samples the model fits its training clips far better than new ones: it memorises particular phases instead of learning the tones, and its test accuracy stays near the 33% of guessing. On log-mel features, 20 numbers per clip instead of 1,024, it is close to perfect on clips it has never seen. Real speech systems still learn from features like these (or learn their own filterbank from the waveform with a convolutional front end), but they are trained on thousands of hours of recorded speech rather than three generated tones.

## Exercises

These go a little beyond the type-alongs. Each has a check cell; the reference is hidden above it.

### Exercise 1: `frame_count`

Write `frame_count(n_samples, frame_length, hop)` that returns the number of complete frames as a Python `int`, or `0` when the signal is shorter than one frame. Use it to answer: how many frames does a 3 s clip at 16 kHz give with a 400-sample window and a 160-sample hop?

In [ ]:
def frame_count_reference(n_samples, frame_length, hop):
    return 0 if n_samples < frame_length else 1 + (n_samples - frame_length) // hop

In [ ]:
def frame_count(n_samples, frame_length, hop):
    """The number of complete frames, or 0 if the signal is shorter than one frame."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
frame_count = practice("frame_count", frame_count, frame_count_reference, cases=[
    (8000, 256, 128), (48000, 400, 160), (100, 256, 128), (256, 256, 128),
])

### Exercise 2: `hz_to_bin`

Write `hz_to_bin(frequency, n_fft, sample_rate)` that returns the index of the FFT bin nearest to `frequency`, as an `int`. Which bin holds 1 kHz for 256-sample frames at 8 kHz, and which for 400-sample frames at 16 kHz?

In [ ]:
def hz_to_bin_reference(frequency, n_fft, sample_rate):
    return int(round(frequency * n_fft / sample_rate))

In [ ]:
def hz_to_bin(frequency, n_fft, sample_rate):
    """The nearest FFT bin to a frequency in hertz."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
hz_to_bin = practice("hz_to_bin", hz_to_bin, hz_to_bin_reference, cases=[
    (1000.0, 256, 8000), (1000.0, 400, 16000), (0.0, 256, 8000), (4000.0, 256, 8000),
])

### Exercise 3: `pre_emphasis`

Many speech front ends boost high frequencies before framing with a **pre-emphasis** filter, $y[0] = x[0]$ and $y[n] = x[n] - a\, x[n-1]$ for $n \ge 1$, usually with $a = 0.97$. Write `pre_emphasis(x, coefficient=0.97)` without a Python loop. Then compare the spectra of a 200 Hz tone and a 3 kHz tone before and after: which one loses more power?

In [ ]:
def pre_emphasis_reference(x, coefficient=0.97):
    return np.concatenate([x[:1], x[1:] - coefficient * x[:-1]])

In [ ]:
def pre_emphasis(x, coefficient=0.97):
    """y[0] = x[0]; y[n] = x[n] - coefficient * x[n - 1]."""
    # ✍️ Your code here (one line with slices)
    raise NotImplementedError

In [ ]:
pre_emphasis = practice("pre_emphasis", pre_emphasis, pre_emphasis_reference, cases=[
    (np.array([1.0, 2.0, 3.0, 4.0]),), (np.ones(5), 0.5), (np.array([2.0]),),
])

### Exercise 4: an open question

Build `mel_filterbank(80, 256, 8000)` and count the filters whose weights are all zero (`(bank.max(axis=1) == 0).sum()`). Why do some filters come out empty, and which two settings could you change to fix it? (Hint: compare the width of the narrowest triangle with the spacing between FFT bins.)

## Common errors and debugging

- **The tone has the wrong pitch.** Check that the angle uses `n / sample_rate` (seconds), not `n`, and radians, not degrees.
- **`operands could not be broadcast together` in the filterbank.** `left`, `centre` and `right` need `[:, None]` so that they form columns against the row of bin frequencies.
- **`-inf` in the features.** A silent frame has zero power; add `eps` before taking the logarithm.
- **A spectrum twice as long as expected.** `np.fft.fft` returns all $N$ bins, including the mirrored half; use `np.fft.rfft` for real signals.
- **Frame counts that disagree with a library by one or two.** The library probably centres frames by padding the signal (often called `center=True`). Match the convention the model was trained with.
- **Pitch and duration both changed after "converting" a file.** The rate was relabelled rather than resampled.
- **Suspiciously good results.** Normalisation statistics computed on all the data, test clips included, leak information. Compute them on the training clips only, as `train_softmax_regression` does.

## Summary

A waveform is samples at a known rate: duration is samples divided by rate, and frequencies above half the rate fold back as aliases. The DFT measures how much of each frequency a block contains; the STFT does this frame by frame, with a window whose length trades time detail against frequency detail. A mel filterbank pools the power into perceptually spaced bands, and a logarithm compresses it. These features remove the phase, which is exactly what made the same linear model go from near chance on raw samples to near-perfect accuracy on log-mel features.

## Next

**02 · CTC recognition**: turn these features into a recogniser that transcribes a sequence of tones without ever being told where each tone starts. You will build the CTC collapse rule, its forward algorithm and its gradient, and train a model with your own CTC loss.